# Final V2 Tuning on Canonical Elo Only

After the ablations, tuning is restricted to the six canonical Elo features so discarded noise cannot re-enter the model.

# כוונון סופי של V2 על Elo קנוני בלבד

מחקרי ההסרה הראו ש־DNA, מפגשים ישירים וכושר מתגלגל אינם משפרים את אות ה־Elo הקנוני בתצורה הנוכחית. לכן המחברת הסופית מסירה אותם ממטריצת האימון ומשאירה רק ארבעה דירוגים נקודתיים בזמן ושני הפרשי דירוגים. הפחתת המרחב מצמצמת הזדמנויות לפיצולים על רעש ומקלה על פירוש תוצאת הכוונון.

In [1]:
from pathlib import Path
import sys

import numpy as np
import optuna
import pandas as pd
from optuna.samplers import TPESampler
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss
from xgboost import XGBClassifier

project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent
elo_module_dir = project_root / "src" / "features"
if str(elo_module_dir) not in sys.path:
    sys.path.insert(0, str(elo_module_dir))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from elo import PointInTimeEngine
from src.data.splitter import ChronologicalSplitter

RANDOM_SEED = 42

## Point-in-Time Elo and Locked Time Split

Canonical identities and deterministic chronological ordering produce pre-match features before the fixed train and validation boundaries are applied.

## בניית Elo נקודתי בזמן וחלוקת זמן נעולה

המנוע משתמש במפתחות הקבוצה הקנוניים וממיין לפי `datetime`, ולאחר מכן לפי `match_id` ו־`game_id` כדי לקבוע סדר מפות בתוך סדרה. דירוגי Global ו־Map נשמרים לפני עדכון התוצאה הנוכחית. המנוע ממשיך לחשב H2H ו־Rolling Form לשימוש עתידי, אך עמודות אלה אינן נכללות ב־allowlist של המודל.

לאחר יצירת הפיצ'רים מתבצעת החלוקה הכרונולוגית: Train עד סוף 2025, Validation ברבעון הראשון של 2026 ו־Test מאפריל 2026. רק Train ו־Validation עוברים סימטריזציה; Test נשאר נעול.

In [2]:
source_columns = [
    "match_id", "game_id", "team1_id", "team1",
    "team2_id", "team2", "is_total", "bestOf",
    "score1_game", "score2_game", "map_name", "datetime",
    "team1_win", "team1_join_key", "team2_join_key",
]
source_df = pd.read_csv(
    project_root / "data" / "final_tournament_features.csv",
    usecols=source_columns, low_memory=False,
)
point_in_time_df = PointInTimeEngine(k_factor=24, initial_rating=1500).transform(source_df)
point_in_time_df["map_position"] = (
    point_in_time_df.groupby("match_id", sort=False).cumcount() + 1
)
splits = ChronologicalSplitter().split(point_in_time_df)
train_base = splits["train"]
val_base = splits["val"]
locked_test_rows = len(splits["test"])

print(f"Train גולמי: {len(train_base):,}")
print(f"Validation גולמי: {len(val_base):,}")
print(f"Test נעול: {locked_test_rows:,}")
print(f"מפה 1 ב־Validation: {val_base['map_position'].eq(1).sum():,}")
print(f"מפות 2+ ב־Validation: {val_base['map_position'].gt(1).sum():,}")

Train גולמי: 5,472
Validation גולמי: 633
Test נעול: 595
מפה 1 ב־Validation: 274
מפות 2+ ב־Validation: 359


## Symmetrization and Minimal Allowlist

Teams are mirrored only after splitting, and the matrix contains four absolute ratings plus global and map Elo differences.

## סימטריזציה ו־allowlist מצומצם

לכל מפה נוצר עותק שבו הקבוצות ודירוגיהן מוחלפים והתווית מתהפכת. ההפרשים מחושבים לאחר ההחלפה, ובדיקה אנטי־סימטרית מחייבת היפוך סימן מדויק. הפעולה מתבצעת באופן עצמאי בכל חלוקה ורק לאחר הפיצול בזמן.

מטריצת האימון כוללת בדיוק שישה פיצ'רים: דירוג Global ודירוג Map לכל צד, והפרש קבוצה 1 פחות קבוצה 2 לכל אחד משני סוגי הדירוג.

In [3]:
rating_pairs = [
    ("team1_elo_global", "team2_elo_global"),
    ("team1_elo_map", "team2_elo_map"),
]
identity_pairs = [
    ("team1_id", "team2_id"),
    ("team1", "team2"),
    ("team1_join_key", "team2_join_key"),
]
absolute_columns = [column for pair in rating_pairs for column in pair]
feature_columns = absolute_columns + ["elo_global_diff", "elo_map_diff"]
context_columns = [
    "match_id", "game_id", "datetime", "map_name", "map_position",
    "team1_id", "team1", "team2_id", "team2",
    "team1_join_key", "team2_join_key", "team1_win",
]

def symmetrize_elo(split_df):
    original = split_df[context_columns + absolute_columns].copy().reset_index(drop=True)
    mirrored = original.copy()
    for team1_column, team2_column in identity_pairs + rating_pairs:
        mirrored[team1_column] = original[team2_column].to_numpy(copy=True)
        mirrored[team2_column] = original[team1_column].to_numpy(copy=True)
    mirrored["team1_win"] = 1 - original["team1_win"].to_numpy()

    symmetric = pd.concat([original, mirrored], ignore_index=True)
    symmetric["elo_global_diff"] = (
        symmetric["team1_elo_global"] - symmetric["team2_elo_global"]
    )
    symmetric["elo_map_diff"] = (
        symmetric["team1_elo_map"] - symmetric["team2_elo_map"]
    )

    midpoint = len(original)
    for diff_column in ["elo_global_diff", "elo_map_diff"]:
        original_values = symmetric.iloc[:midpoint][diff_column].to_numpy()
        mirrored_values = symmetric.iloc[midpoint:][diff_column].to_numpy()
        if not np.allclose(original_values, -mirrored_values):
            raise AssertionError(f"Diff sign did not flip: {diff_column}")
    if not np.isclose(symmetric["team1_win"].mean(), 0.5):
        raise AssertionError("Symmetrization did not balance the target.")
    return symmetric

train_df = symmetrize_elo(train_base)
val_df = symmetrize_elo(val_base)
X_train = train_df[feature_columns]
y_train = train_df["team1_win"].astype(int)
X_val = val_df[feature_columns]
y_val = val_df["team1_win"].astype(int)

print(f"פיצ'רים מאושרים: {feature_columns}")
print(f"Train מסומטר: {len(train_df):,}")
print(f"Validation מסומטר: {len(val_df):,}")

פיצ'רים מאושרים: ['team1_elo_global', 'team2_elo_global', 'team1_elo_map', 'team2_elo_map', 'elo_global_diff', 'elo_map_diff']
Train מסומטר: 10,944
Validation מסומטר: 1,266


## Reproducing the Fixed Baseline

The untuned Elo-only model is refit first so the tuned candidate is compared against a locally reproduced baseline.

## שחזור קו הבסיס הקבוע

לפני הכוונון מאומן מחדש מודל Elo-only בתצורה הקבועה של מחקרי ההסרה. התחזיות שלו נשמרות כדי לאפשר Bootstrap מזווג מול המודל המכוונן על אותן מפות ואותם אשכולות משחק. המדדים המתועדים משמשים גם בדיקת שחזור מול הערכים 0.608 דיוק, 0.582 דיוק במפה 1, 0.236 ברייר כללי ו־0.239 ברייר במפה 1.

In [4]:
fixed_model = XGBClassifier(
    objective="binary:logistic", eval_metric="logloss",
    n_estimators=1000, learning_rate=0.03, max_depth=3,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.9,
    reg_lambda=2.0, tree_method="hist", random_state=RANDOM_SEED,
    n_jobs=-1, early_stopping_rounds=40,
)
fixed_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=False,
)
fixed_probability = fixed_model.predict_proba(X_val)[:, 1]
fixed_prediction = (fixed_probability >= 0.5).astype(int)
map1_mask = val_df["map_position"].eq(1)
fixed_accuracy = accuracy_score(y_val, fixed_prediction)
fixed_brier = brier_score_loss(y_val, fixed_probability)
fixed_map1_accuracy = accuracy_score(
    y_val.loc[map1_mask], fixed_prediction[map1_mask.to_numpy()]
)
fixed_map1_brier = brier_score_loss(
    y_val.loc[map1_mask], fixed_probability[map1_mask.to_numpy()]
)

print(f"דיוק קו בסיס: {fixed_accuracy:.6f}")
print(f"ברייר קו בסיס: {fixed_brier:.6f}")
print(f"דיוק קו בסיס במפה 1: {fixed_map1_accuracy:.6f}")
print(f"ברייר קו בסיס במפה 1: {fixed_map1_brier:.6f}")

דיוק קו בסיס: 0.608215
ברייר קו בסיס: 0.236055
דיוק קו בסיס במפה 1: 0.582117
ברייר קו בסיס במפה 1: 0.238658


## TPE Optimization for Validation Log-Loss

Fifty seeded Optuna trials search depth, learning rate, sampling, and regularization, with validation log-loss as the objective.

## כוונון TPE לפי לוג־לוס

Optuna מריץ 50 ניסויים עם דוגם TPE בעל seed קבוע. כל ניסוי בוחר עומק, קצב למידה, שיעורי דגימת שורות ועמודות, וענישות L1/L2. קצב הלמידה ושני פרמטרי הענישה נדגמים בסולם לוגריתמי מפני שהבדלים יחסיים חשובים יותר מהבדלים מוחלטים בטווחים אלה.

פונקציית המטרה ממזערת לוג־לוס על Validation. בכל ניסוי עצירה מוקדמת קובעת כמה עצים נדרשים בפועל. Test אינו משתתף באימון, בעצירה, בבחירת פרמטרים או בדיווח.

In [5]:
def build_tuned_model(parameters):
    return XGBClassifier(
        objective="binary:logistic", eval_metric="logloss",
        n_estimators=1200, tree_method="hist",
        random_state=RANDOM_SEED, n_jobs=-1, early_stopping_rounds=50,
        **parameters,
    )

def objective(trial):
    parameters = {
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
    }
    model = build_tuned_model(parameters)
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_val, y_val)],
        verbose=False,
    )
    probability = model.predict_proba(X_val)[:, 1]
    trial.set_user_attr("best_iteration", int(model.best_iteration))
    return log_loss(y_val, probability, labels=[0, 1])

study = optuna.create_study(
    direction="minimize",
    sampler=TPESampler(seed=RANDOM_SEED, n_startup_trials=10),
)
study.optimize(objective, n_trials=50, gc_after_trial=True, show_progress_bar=False)

print(f"מספר ניסויים שהושלמו: {len(study.trials)}")
print(f"לוג־לוס מיטבי בחיפוש: {study.best_value:.6f}")
print("פרמטרים נבחרים:")
for parameter_name, parameter_value in study.best_params.items():
    print(f"  {parameter_name}: {parameter_value}")

[I 2026-09-15 09:44:44,100] A new study created in memory with name: no-name-6f656b56-5f70-4896-9edd-728a409af556


[I 2026-09-15 09:44:48,872] Trial 0 finished with value: 0.6664543747901917 and parameters: {'max_depth': 5, 'learning_rate': 0.2536999076681772, 'subsample': 0.892797576724562, 'colsample_bytree': 0.8394633936788146, 'reg_alpha': 1.77071686435378e-07, 'reg_lambda': 1.7699302940633311e-07}. Best is trial 0 with value: 0.6664543747901917.


[I 2026-09-15 09:45:04,017] Trial 1 finished with value: 0.6633148193359375 and parameters: {'max_depth': 3, 'learning_rate': 0.19030368381735815, 'subsample': 0.8404460046972835, 'colsample_bytree': 0.8832290311184181, 'reg_alpha': 1.4610865886287176e-08, 'reg_lambda': 0.574485163632042}. Best is trial 1 with value: 0.6633148193359375.


[I 2026-09-15 09:45:25,828] Trial 2 finished with value: 0.6665210723876953 and parameters: {'max_depth': 7, 'learning_rate': 0.020589728197687916, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735, 'reg_alpha': 2.716051144654844e-06, 'reg_lambda': 0.00015777981883364995}. Best is trial 1 with value: 0.6633148193359375.


[I 2026-09-15 09:45:31,058] Trial 3 finished with value: 0.6655552387237549 and parameters: {'max_depth': 5, 'learning_rate': 0.02692655251486473, 'subsample': 0.8447411578889518, 'colsample_bytree': 0.6557975442608167, 'reg_alpha': 2.1734877073417355e-06, 'reg_lambda': 8.528933855762793e-06}. Best is trial 1 with value: 0.6633148193359375.


...

[I 2026-09-15 09:47:10,246] Trial 46 finished with value: 0.6640492081642151 and parameters: {'max_depth': 3, 'learning_rate': 0.2953389008717481, 'subsample': 0.9252810835957178, 'colsample_bytree': 0.9434685437704348, 'reg_alpha': 2.2847538122422806e-08, 'reg_lambda': 2.1519465317257883e-08}. Best is trial 44 with value: 0.6630033850669861.


[I 2026-09-15 09:47:12,646] Trial 47 finished with value: 0.664476752281189 and parameters: {'max_depth': 3, 'learning_rate': 0.20615289991235897, 'subsample': 0.965305367886234, 'colsample_bytree': 0.8820887065916242, 'reg_alpha': 4.2326241516311835e-06, 'reg_lambda': 2.3676693520653953e-08}. Best is trial 44 with value: 0.6630033850669861.


[I 2026-09-15 09:47:14,068] Trial 48 finished with value: 0.6643006205558777 and parameters: {'max_depth': 4, 'learning_rate': 0.2773235099148248, 'subsample': 0.9392086258302176, 'colsample_bytree': 0.9085193206342885, 'reg_alpha': 4.691509111261365e-07, 'reg_lambda': 1.772393814708473e-08}. Best is trial 44 with value: 0.6630033850669861.


[I 2026-09-15 09:47:16,155] Trial 49 finished with value: 0.6641484498977661 and parameters: {'max_depth': 3, 'learning_rate': 0.21560061711680267, 'subsample': 0.9608098034926944, 'colsample_bytree': 0.9654341192493555, 'reg_alpha': 7.100173644959831e-08, 'reg_lambda': 3.6921734883734123e-07}. Best is trial 44 with value: 0.6630033850669861.


מספר ניסויים שהושלמו: 50
לוג־לוס מיטבי בחיפוש: 0.663003
פרמטרים נבחרים:
  max_depth: 3
  learning_rate: 0.22473870152166872
  subsample: 0.9744377938554797
  colsample_bytree: 0.9620639280745505
  reg_alpha: 1.4445201932091805e-06
  reg_lambda: 1.6603528779515085e-05


## Refit and Validation Stratification

The selected configuration is refit on training data and evaluated overall and by map position.

## אימון התצורה הנבחרת ופילוח Validation

המודל הנבחר מאומן מחדש על Train עם עצירה מוקדמת מול Validation. מדווחים לוג־לוס, דיוק וברייר בכלל החלוקה, ולאחר מכן דיוק וברייר במפה 1 ובמפות 2 ומעלה. מיקום המפה משמש רק לפילוח ואינו פיצ'ר.

In [6]:
best_model = build_tuned_model(study.best_params)
best_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=10,
)
tuned_probability = best_model.predict_proba(X_val)[:, 1]
tuned_prediction = (tuned_probability >= 0.5).astype(int)

def calculate_tuned_metrics(mask):
    selected_target = y_val.loc[mask]
    selected_probability = tuned_probability[mask.to_numpy()]
    selected_prediction = tuned_prediction[mask.to_numpy()]
    return {
        "rows": int(mask.sum()),
        "accuracy": accuracy_score(selected_target, selected_prediction),
        "brier": brier_score_loss(selected_target, selected_probability),
    }

metric_groups = {
    "כלל האימות": pd.Series(True, index=val_df.index),
    "מפה 1": val_df["map_position"].eq(1),
    "מפה 2 ומעלה": val_df["map_position"].gt(1),
}
tuned_metrics = {
    group_name: calculate_tuned_metrics(mask)
    for group_name, mask in metric_groups.items()
}
final_log_loss = log_loss(y_val, tuned_probability, labels=[0, 1])
print(f"האיטרציה הטובה ביותר: {best_model.best_iteration}")
print(f"לוג־לוס סופי: {final_log_loss:.6f}")
for group_name, metrics in tuned_metrics.items():
    print(
        f"{group_name}: שורות={metrics['rows']:,}, "
        f"דיוק={metrics['accuracy']:.6f}, ברייר={metrics['brier']:.6f}"
    )

[0]	validation_0-logloss:0.68277	validation_1-logloss:0.67973


[10]	validation_0-logloss:0.66032	validation_1-logloss:0.66583


[20]	validation_0-logloss:0.65295	validation_1-logloss:0.66986


[30]	validation_0-logloss:0.64805	validation_1-logloss:0.67127


[40]	validation_0-logloss:0.64319	validation_1-logloss:0.67507


[50]	validation_0-logloss:0.63877	validation_1-logloss:0.67887


[57]	validation_0-logloss:0.63583	validation_1-logloss:0.67922


האיטרציה הטובה ביותר: 7
לוג־לוס סופי: 0.663003
כלל האימות: שורות=1,266, דיוק=0.610585, ברייר=0.235418
מפה 1: שורות=548, דיוק=0.589416, ברייר=0.238312
מפה 2 ומעלה: שורות=718, דיוק=0.626741, ברייר=0.233209


## Paired Cluster-Bootstrap Guardrail

Both models predict the same rows, so matches are resampled jointly and paired metric differences preserve within-match dependence.

## Guardrail סטטיסטי באתחול מחדש מזווג

שני המודלים מייצרים תחזית לאותן שורות, ולכן בכל חזרת Bootstrap נדגמים 274 אשכולות `match_id` עם החזרה וכל השורות של המשחק נשמרות יחד. השוואה מזווגת זו מבטלת חלק מהשונות המשותפת ומודדת ישירות את השינוי שנוצר מהכוונון.

פער הדיוק מוגדר כדיוק המודל המכוונן פחות דיוק המודל הקבוע. שיפור הברייר מוגדר כברייר המודל הקבוע פחות ברייר המודל המכוונן. ערך חיובי מעדיף את הכוונון. החישוב נעשה הן בכלל Validation והן במפה 1, שהיא שכבת ה־pre-series המחמירה.

In [7]:
bootstrap_iterations = 1000
bootstrap_rng = np.random.default_rng(RANDOM_SEED)
target_values = y_val.to_numpy()
map1_values = map1_mask.to_numpy()
tuned_correct = (tuned_prediction == target_values).astype(float)
fixed_correct = (fixed_prediction == target_values).astype(float)
tuned_squared_error = (tuned_probability - target_values) ** 2
fixed_squared_error = (fixed_probability - target_values) ** 2

bootstrap_rows = pd.DataFrame({
    "match_id": val_df["match_id"].to_numpy(),
    "rows": 1.0,
    "tuned_correct": tuned_correct,
    "fixed_correct": fixed_correct,
    "tuned_brier_sum": tuned_squared_error,
    "fixed_brier_sum": fixed_squared_error,
    "map1_rows": map1_values.astype(float),
    "map1_tuned_correct": tuned_correct * map1_values,
    "map1_fixed_correct": fixed_correct * map1_values,
    "map1_tuned_brier_sum": tuned_squared_error * map1_values,
    "map1_fixed_brier_sum": fixed_squared_error * map1_values,
})
cluster_metrics = bootstrap_rows.groupby("match_id", sort=False).sum()
cluster_values = cluster_metrics.to_numpy(dtype=float)
match_count = len(cluster_values)
sample_indices = bootstrap_rng.integers(
    0, match_count, size=(bootstrap_iterations, match_count)
)
sample_totals = cluster_values[sample_indices].sum(axis=1)
rows = sample_totals[:, 0]
map1_rows = sample_totals[:, 5]

tuned_accuracy_samples = sample_totals[:, 1] / rows
tuned_brier_samples = sample_totals[:, 3] / rows
overall_accuracy_delta = (sample_totals[:, 1] - sample_totals[:, 2]) / rows
overall_brier_improvement = (sample_totals[:, 4] - sample_totals[:, 3]) / rows
map1_tuned_accuracy_samples = sample_totals[:, 6] / map1_rows
map1_tuned_brier_samples = sample_totals[:, 8] / map1_rows
map1_accuracy_delta = (sample_totals[:, 6] - sample_totals[:, 7]) / map1_rows
map1_brier_improvement = (sample_totals[:, 9] - sample_totals[:, 8]) / map1_rows

def confidence_interval(values):
    return np.quantile(values, [0.025, 0.975])

overall_accuracy_ci = confidence_interval(tuned_accuracy_samples)
overall_brier_ci = confidence_interval(tuned_brier_samples)
overall_accuracy_delta_ci = confidence_interval(overall_accuracy_delta)
overall_brier_improvement_ci = confidence_interval(overall_brier_improvement)
map1_accuracy_ci = confidence_interval(map1_tuned_accuracy_samples)
map1_brier_ci = confidence_interval(map1_tuned_brier_samples)
map1_accuracy_delta_ci = confidence_interval(map1_accuracy_delta)
map1_brier_improvement_ci = confidence_interval(map1_brier_improvement)

print(f"מספר משחקים ייחודיים: {match_count}; חזרות Bootstrap: {bootstrap_iterations}")
print(f"דיוק מכוונן — רווח סמך 95%: [{overall_accuracy_ci[0]:.6f}, {overall_accuracy_ci[1]:.6f}]")
print(f"ברייר מכוונן — רווח סמך 95%: [{overall_brier_ci[0]:.6f}, {overall_brier_ci[1]:.6f}]")
print(f"שינוי דיוק מול קו הבסיס — רווח סמך 95%: [{overall_accuracy_delta_ci[0]:.6f}, {overall_accuracy_delta_ci[1]:.6f}]")
print(f"שיפור ברייר מול קו הבסיס — רווח סמך 95%: [{overall_brier_improvement_ci[0]:.6f}, {overall_brier_improvement_ci[1]:.6f}]")
print(f"דיוק מכוונן במפה 1 — רווח סמך 95%: [{map1_accuracy_ci[0]:.6f}, {map1_accuracy_ci[1]:.6f}]")
print(f"ברייר מכוונן במפה 1 — רווח סמך 95%: [{map1_brier_ci[0]:.6f}, {map1_brier_ci[1]:.6f}]")
print(f"שינוי דיוק במפה 1 — רווח סמך 95%: [{map1_accuracy_delta_ci[0]:.6f}, {map1_accuracy_delta_ci[1]:.6f}]")
print(f"שיפור ברייר במפה 1 — רווח סמך 95%: [{map1_brier_improvement_ci[0]:.6f}, {map1_brier_improvement_ci[1]:.6f}]")

מספר משחקים ייחודיים: 274; חזרות Bootstrap: 1000
דיוק מכוונן — רווח סמך 95%: [0.572556, 0.649594]
ברייר מכוונן — רווח סמך 95%: [0.225775, 0.245452]
שינוי דיוק מול קו הבסיס — רווח סמך 95%: [-0.006280, 0.011528]
שיפור ברייר מול קו הבסיס — רווח סמך 95%: [-0.000499, 0.001800]
דיוק מכוונן במפה 1 — רווח סמך 95%: [0.534672, 0.653285]
ברייר מכוונן במפה 1 — רווח סמך 95%: [0.224706, 0.250949]
שינוי דיוק במפה 1 — רווח סמך 95%: [-0.005474, 0.023723]
שיפור ברייר במפה 1 — רווח סמך 95%: [-0.001302, 0.001992]


## Tuning Decision

Because the tuned model does not statistically clear the fixed baseline, production retains the simpler default canonical-Elo configuration.

## מסקנת הכוונון ונקודת החלטה

המודל המכוונן השיג לוג־לוס 0.663003, דיוק 0.610585 וברייר 0.235418 בכלל Validation. במפה 1 התקבלו דיוק 0.589416 וברייר 0.238312; במפות 2+ התקבלו דיוק 0.626741 וברייר 0.233209. לעומת התצורה הקבועה, השיפור הנקודתי הוא 0.002370 בדיוק ו־0.000637 בברייר בכלל החלוקה, ו־0.007299 בדיוק ו־0.000346 בברייר במפה 1.

רווח הסמך המזווג לשינוי הדיוק הכולל הוא ‎[-0.006280, 0.011528]‎ ורווח הסמך לשיפור הברייר הכולל הוא ‎[-0.000499, 0.001800]‎. במפה 1 הרווחים הם ‎[-0.005474, 0.023723]‎ לדיוק ו־‎[-0.001302, 0.001992]‎ לברייר. כל ארבעת הרווחים כוללים אפס, ולכן הכוונון אינו מנקה סטטיסטית את קו הבסיס הקבוע במדגם זה. התצורה הנבחרת נשמרת כמועמדת ניסויית בלבד; Test נשאר ללא סימטריזציה, ללא חיזוי וללא הערכה.